# DINO Feature Geometry

Compare within-class and across-class similarity. Does every pair follow the average?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://github.com/facebookresearch/dino

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "timm==1.0.22"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

import timm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
# Load the ViT-Small model trained with DINO
model_name = 'vit_small_patch16_224.dino'
model = timm.create_model(model_name, pretrained=True, num_classes=0)  # num_classes=0 → feature extractor
model = model.to(device)
model.eval()

print(f"✅ Model loaded: {model_name}")
print(f"   - Patch size: 16x16")
print(f"   - Input size: 224x224")
print(f"   - Embedding dim: {model.embed_dim}")
print(f"   - Number of heads: {model.blocks[0].attn.num_heads}")
print(f"   - Number of layers: {len(model.blocks)}")

# Check the number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"   - Total parameters: {total_params / 1e6:.1f}M")

In [ ]:
# Use the model-specific transform provided by timm
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

print("📐 Transform pipeline:")
print(transform)

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
class DogCatDataset(Dataset):
    """Dog vs Cat image dataset"""
    def __init__(self, img_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.class_names = ['cat', 'dog']
        
        for label_idx, label in enumerate(self.class_names):
            class_dir = os.path.join(img_dir, label)
            if not os.path.exists(class_dir):
                continue
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(class_dir, fname), label_idx))
        
        print(f"   Dataset: {len(self.samples)} images ({self.class_names})")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, path

print("📂 Building dataset:")
dataset = DogCatDataset(IMG_DIR, transform=transform)

In [ ]:
@torch.no_grad()
def extract_features(model, dataset, device, batch_size=16):
    """
    Extract image features with DINO ViT.
    The CLS token output is used as the feature vector.
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    all_features = []
    all_labels = []
    all_paths = []
    
    model.eval()
    for images, labels, paths in tqdm(loader, desc='Extracting features'):
        images = images.to(device)
        features = model(images)  # (B, embed_dim) — CLS token output
        
        all_features.append(features.cpu().numpy())
        all_labels.append(labels.numpy())
        all_paths.extend(paths)
    
    features = np.concatenate(all_features, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    
    print(f"\n✅ Feature extraction complete!")
    print(f"   - Feature shape: {features.shape}")
    print(f"   - Labels shape: {labels.shape}")
    
    return features, labels, all_paths

features, labels, paths = extract_features(model, dataset, device)

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    features, labels, paths,
    test_size=0.3,
    random_state=SEED,
    stratify=labels
)

print(f"📊 Train/Test Split:")
print(f"   Train: {len(X_train)} samples (Dog: {(y_train==1).sum()}, Cat: {(y_train==0).sum()})")
print(f"   Test:  {len(X_test)} samples (Dog: {(y_test==1).sum()}, Cat: {(y_test==0).sum()})")

## Demo code

Compare within-class and across-class similarity. Does every pair follow the average?

In [ ]:
unit_features = features / np.linalg.norm(features, axis=1, keepdims=True)
similarity = unit_features @ unit_features.T
query = 0
neighbors = np.argsort(similarity[query])[::-1][1:6]
print([(paths[i], similarity[query, i]) for i in neighbors])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Normalize features
features_norm = features / np.linalg.norm(features, axis=1, keepdims=True)

# Full cosine similarity matrix
cos_sim_matrix = cosine_similarity(features_norm)

# Separate by class
cat_idx = np.where(labels == 0)[0]
dog_idx = np.where(labels == 1)[0]

# Compute intra-class / inter-class similarity
cat_cat_sim = cos_sim_matrix[np.ix_(cat_idx, cat_idx)]
dog_dog_sim = cos_sim_matrix[np.ix_(dog_idx, dog_idx)]
cat_dog_sim = cos_sim_matrix[np.ix_(cat_idx, dog_idx)]

# Remove the diagonal (self)
cat_cat_vals = cat_cat_sim[np.triu_indices_from(cat_cat_sim, k=1)]
dog_dog_vals = dog_dog_sim[np.triu_indices_from(dog_dog_sim, k=1)]
cat_dog_vals = cat_dog_sim.flatten()

print(f"📊 Cosine Similarity Statistics:")
print(f"{'='*50}")
print(f"  Cat ↔ Cat  (intra-class):  {cat_cat_vals.mean():.4f} ± {cat_cat_vals.std():.4f}")
print(f"  Dog ↔ Dog  (intra-class):  {dog_dog_vals.mean():.4f} ± {dog_dog_vals.std():.4f}")
print(f"  Cat ↔ Dog  (inter-class):  {cat_dog_vals.mean():.4f} ± {cat_dog_vals.std():.4f}")
print(f"{'='*50}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# --- (1) Similarity histogram ---
axes[0].hist(cat_cat_vals, bins=30, alpha=0.6, label='Cat↔Cat', color='#FF6B6B', density=True)
axes[0].hist(dog_dog_vals, bins=30, alpha=0.6, label='Dog↔Dog', color='#4ECDC4', density=True)
axes[0].hist(cat_dog_vals, bins=30, alpha=0.6, label='Cat↔Dog', color='#95A5A6', density=True)
axes[0].set_xlabel('Cosine Similarity', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Similarity Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].axvline(x=cat_dog_vals.mean(), color='gray', linestyle='--', alpha=0.7)

# --- (2) Similarity Heatmap (sorted) ---
sorted_idx = np.concatenate([cat_idx, dog_idx])
sorted_sim = cos_sim_matrix[np.ix_(sorted_idx, sorted_idx)]
im = axes[1].imshow(sorted_sim, cmap='RdYlBu_r', vmin=0, vmax=1)
axes[1].set_title('Cosine Similarity Matrix\n(sorted by class)', fontsize=14, fontweight='bold')
n_cats = len(cat_idx)
axes[1].axhline(y=n_cats-0.5, color='black', linewidth=2)
axes[1].axvline(x=n_cats-0.5, color='black', linewidth=2)
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Sample Index')
plt.colorbar(im, ax=axes[1], shrink=0.8)

# --- (3) Box Plot ---
bp = axes[2].boxplot(
    [cat_cat_vals, dog_dog_vals, cat_dog_vals],
    tick_labels=['Cat↔Cat', 'Dog↔Dog', 'Cat↔Dog'],
    patch_artist=True,
    boxprops=dict(linewidth=2),
    medianprops=dict(linewidth=2, color='black')
)
colors = ['#FF6B6B', '#4ECDC4', '#95A5A6']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_ylabel('Cosine Similarity', fontsize=12)
axes[2].set_title('Similarity Comparison', fontsize=14, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Observation: similarity within the same class is higher than across classes!")
print("   → Shows that DINO learned meaningful representations even without labels.")

In [ ]:
# Compare image pairs from the same class vs different classes
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🔗 Pairwise Cosine Similarity Examples', fontsize=16, fontweight='bold')

pairs = [
    # (idx1, idx2, description)
    (cat_idx[0], cat_idx[1], 'Cat vs Cat'),
    (dog_idx[0], dog_idx[1], 'Dog vs Dog'),
    (cat_idx[0], dog_idx[0], 'Cat vs Dog'),
]

for col, (i, j, desc) in enumerate(pairs):
    sim = cos_sim_matrix[i, j]
    
    img1 = Image.open(paths[i]).convert('RGB')
    img2 = Image.open(paths[j]).convert('RGB')
    
    axes[0, col].imshow(img1)
    axes[0, col].set_title(f'Image A', fontsize=11)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(img2)
    axes[1, col].set_title(f'Image B', fontsize=11)
    axes[1, col].axis('off')
    
    color = '#4CAF50' if sim > 0.7 else '#FF9800' if sim > 0.5 else '#F44336'
    fig.text(0.17 + col * 0.33, 0.48, f'{desc}\nSimilarity: {sim:.4f}', 
             ha='center', fontsize=14, fontweight='bold', color=color,
             bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor=color))

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.subplots_adjust(hspace=0.4)
plt.show()

In [ ]:
print("🔄 Computing t-SNE... (please wait a moment)")

tsne = TSNE(
    n_components=2,
    perplexity=min(30, len(features) - 1),
    random_state=SEED,
    max_iter=1000,
    learning_rate='auto',
    init='pca'
)

embeddings_2d = tsne.fit_transform(features_norm)
print(f"✅ t-SNE done! Shape: {embeddings_2d.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- (1) Basic t-SNE plot ---
colors_map = {0: '#FF6B6B', 1: '#4ECDC4'}
markers_map = {0: 'o', 1: 's'}
label_map = {0: 'Cat 🐱', 1: 'Dog 🐶'}

for cls in [0, 1]:
    mask = labels == cls
    axes[0].scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=colors_map[cls], marker=markers_map[cls],
        s=80, alpha=0.7, edgecolors='white', linewidth=0.5,
        label=label_map[cls]
    )
axes[0].set_title('t-SNE of DINO Features', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=13, loc='best')
axes[0].set_xlabel('t-SNE 1', fontsize=12)
axes[0].set_ylabel('t-SNE 2', fontsize=12)
axes[0].grid(alpha=0.2)

# --- (2) Show Train/Test split ---
# Map Train/Test indices
train_mask = np.zeros(len(labels), dtype=bool)
test_mask = np.zeros(len(labels), dtype=bool)

# Find train/test indices based on paths
paths_set_train = set(paths_train)
paths_set_test = set(paths_test)
for i, p in enumerate(paths):
    if p in paths_set_train:
        train_mask[i] = True
    elif p in paths_set_test:
        test_mask[i] = True

for cls in [0, 1]:
    # Train
    mask = (labels == cls) & train_mask
    axes[1].scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=colors_map[cls], marker='o',
        s=60, alpha=0.5, edgecolors='white', linewidth=0.5,
        label=f'{label_map[cls]} (train)'
    )
    # Test
    mask = (labels == cls) & test_mask
    axes[1].scatter(
        embeddings_2d[mask, 0], embeddings_2d[mask, 1],
        c=colors_map[cls], marker='*',
        s=200, alpha=0.9, edgecolors='black', linewidth=1,
        label=f'{label_map[cls]} (test)'
    )

axes[1].set_title('t-SNE with Train/Test Split', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10, loc='best')
axes[1].set_xlabel('t-SNE 1', fontsize=12)
axes[1].set_ylabel('t-SNE 2', fontsize=12)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("\n💡 Observation: Cat and Dog are well separated in t-SNE space!")
print("   → Self-supervised learning alone learns features useful for class separation.")

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

fig, ax = plt.subplots(figsize=(14, 14))
ax.set_title('t-SNE with Image Thumbnails', fontsize=16, fontweight='bold')

# Scatter in the background
for cls in [0, 1]:
    mask = labels == cls
    ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
              c=colors_map[cls], alpha=0.15, s=200)

# Show image thumbnails (only some)
n_thumbnails = min(30, len(paths))
thumb_indices = np.linspace(0, len(paths)-1, n_thumbnails, dtype=int)

for idx in thumb_indices:
    try:
        img = Image.open(paths[idx]).convert('RGB').resize((40, 40))
        img_arr = np.array(img)
        
        im = OffsetImage(img_arr, zoom=0.7)
        border_color = colors_map[labels[idx]]
        ab = AnnotationBbox(im, (embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
                           frameon=True, pad=0.1,
                           bboxprops=dict(edgecolor=border_color, linewidth=2))
        ax.add_artist(ab)
    except:
        pass

ax.grid(alpha=0.2)
ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
plt.tight_layout()
plt.show()

## Interpretation

t-SNE displays geometry; it does not validate a classifier.

Compare the output with the evidence in the input. A generated answer can be wrong.